# MOSAIC — K2 IA Random Forest + XGBoost

This notebook evaluates the **K2 intercept + amplitude (IA)** spectral information domain using two nonlinear supervised models:

- Random Forest
- XGBoost

For each functional group, the ecological response is decomposed into:

1. **Occurrence**: `presence = cover > 0`
2. **Positive abundance**: `cover | cover > 0`

The objective is **feature-space evaluation**, not replacement of the Bayesian ZIB as the primary inferential model.

Primary outputs:

- Cross-validated occurrence AUC / log loss
- Cross-validated positive-abundance R² / RMSE
- Combined hurdle-style expected-cover R²
- Fold-wise held-out permutation importance
- Importance stability across folds
- Spectral-family summaries for correlated predictors

## Expected CSV exported from R

Export one analysis table containing:

- `PlotID`
- `Year`
- the seven functional-group cover responses
- all 45 K2 IA predictors:
  - 15 `constant_*`
  - 15 `amp1_*`
  - 15 `amp2_*`

Using the same standardized predictor table as the ZIB analysis keeps the feature space directly aligned with the Bayesian work. Tree models themselves do not require standardization.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    brier_score_loss,
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn.model_selection import StratifiedKFold

try:
    from xgboost import XGBClassifier, XGBRegressor
except ImportError as e:
    raise ImportError(
        "xgboost is required. Install it with: pip install xgboost"
    ) from e

RANDOM_STATE = 42
N_SPLITS = 5

DATA_CSV = Path(r"C:/NCA_DATA/Analysis/K2_IA_ZIB_export.csv")

FUNCTIONAL_GROUPS = [
    "BAREGROUND_FG",
    "BIOCRUST_FG",
    "EAG_FG",
    "EF_FG",
    "LITTER_FG",
    "PBG_FG",
    "SHRUB_total",
]

In [3]:
# Load and validate the R export

df = pd.read_csv(DATA_CSV)

if "Plot" in df.columns and "PlotID" not in df.columns:
    df = df.rename(columns={"Plot": "PlotID"})

if "PlotID" in df.columns:
    df["PlotID"] = (
        df["PlotID"]
        .astype(str)
        .str.strip()
        .str.replace(r"\\.0$", "", regex=True)
    )

IA_COLS = [
    c for c in df.columns
    if c.lower().startswith((
        "constant_", "intercept_",
        "amp1_", "amp2_",
        "amplitude1_", "amplitude2_"
    ))
]

INTERCEPT_COLS = [
    c for c in IA_COLS
    if c.lower().startswith(("constant_", "intercept_"))
]

AMP1_COLS = [
    c for c in IA_COLS
    if c.lower().startswith(("amp1_", "amplitude1_"))
]

AMP2_COLS = [
    c for c in IA_COLS
    if c.lower().startswith(("amp2_", "amplitude2_"))
]

print(f"Rows: {len(df):,}")
print(f"IA predictors: {len(IA_COLS)}")
print(f"  intercepts: {len(INTERCEPT_COLS)}")
print(f"  H1 amplitudes: {len(AMP1_COLS)}")
print(f"  H2 amplitudes: {len(AMP2_COLS)}")

missing_fg = [fg for fg in FUNCTIONAL_GROUPS if fg not in df.columns]
if missing_fg:
    raise ValueError(f"Missing functional-group columns: {missing_fg}")

if len(IA_COLS) != 45:
    print("WARNING: expected 45 K2 IA predictors. Inspect IA_COLS before proceeding.")

display(df[["PlotID", "Year"] + FUNCTIONAL_GROUPS + IA_COLS[:5]].head())

Rows: 489
IA predictors: 45
  intercepts: 15
  H1 amplitudes: 15
  H2 amplitudes: 15


,PlotID,Year,BAREGROUND_FG,BIOCRUST_FG,EAG_FG,EF_FG,LITTER_FG,PBG_FG,SHRUB_total,constant_B2,constant_B3,constant_B4,constant_B5,constant_B6
0,100,2016,43.4,0.2,0.0,52.6,3.2,0.0,0.0,1.273798,1.389091,1.602432,1.694911,1.474544
1,101,2016,41.4,5.6,8.6,33.6,9.0,0.0,0.0,1.101793,1.187839,1.266428,1.249331,1.196625
2,102,2016,13.2,2.8,68.8,0.0,13.8,1.0,0.0,0.933559,0.907829,0.968814,0.997615,0.900752
3,103,2016,28.8,5.2,42.4,0.0,22.2,0.0,0.0,1.113955,1.139120,1.202900,0.906339,0.748714
4,104,2016,21.4,3.0,53.8,0.0,16.6,0.0,4.2,0.517199,0.421899,0.349338,0.434358,0.371424


## Modeling logic

For functional-group cover \(Y\):

### Occurrence
\[
Z = \mathbb{1}(Y > 0)
\]

A classifier estimates:

\[
\hat p_i = P(Y_i > 0 \mid X_i)
\]

### Positive abundance
For observations with \(Y>0\), a regressor estimates:

\[
\hat \mu_i = E(Y_i \mid Y_i>0, X_i)
\]

### Combined expected cover
For every observation:

\[
\widehat{E(Y_i \mid X_i)} = \hat p_i \hat \mu_i
\]

This mirrors the ecological decomposition used in the hurdle/ZIB work while allowing nonlinear effects and interactions.

In [4]:
MODELS = {
    "RandomForest": {
        "classifier": RandomForestClassifier(
            n_estimators=800,
            max_features="sqrt",
            min_samples_leaf=3,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "regressor": RandomForestRegressor(
            n_estimators=800,
            max_features="sqrt",
            min_samples_leaf=3,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    },

    "XGBoost": {
        "classifier": XGBClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.05,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "regressor": XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.05,
            reg_lambda=1.0,
            objective="reg:squarederror",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    },
}

In [5]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5


def fit_fg_cv(
    data,
    fg,
    feature_cols,
    model_name,
    model_spec,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    permutation_repeats=15,
):
    cols = feature_cols + [fg]
    work = data[cols].dropna().copy()

    X = work[feature_cols].to_numpy()
    y = work[fg].to_numpy(dtype=float)

    # Accept either percent cover or proportion.
    if np.nanmax(y) > 1.0:
        y = y / 100.0

    presence = (y > 0).astype(int)

    n_pos = int(presence.sum())
    n_zero = int(len(presence) - n_pos)

    if n_pos < n_splits or n_zero < n_splits:
        raise ValueError(
            f"{fg}: insufficient positive/zero observations for {n_splits}-fold CV."
        )

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    oof_p_presence = np.full(len(work), np.nan)
    oof_mu_positive = np.full(len(work), np.nan)

    fold_metrics = []
    importance_rows = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, presence), start=1):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        z_train, z_test = presence[train_idx], presence[test_idx]

        # -------------------------------------------------------------
        # Occurrence
        # -------------------------------------------------------------
        clf = clone(model_spec["classifier"])
        clf.fit(X_train, z_train)

        p_test = clf.predict_proba(X_test)[:, 1]
        oof_p_presence[test_idx] = p_test

        auc = roc_auc_score(z_test, p_test)
        ll = log_loss(z_test, p_test, labels=[0, 1])
        brier = brier_score_loss(z_test, p_test)

        pi_occ = permutation_importance(
            clf,
            X_test,
            z_test,
            scoring="roc_auc",
            n_repeats=permutation_repeats,
            random_state=random_state + fold,
            n_jobs=-1,
        )

        for feature, imp_mean, imp_sd in zip(
            feature_cols,
            pi_occ.importances_mean,
            pi_occ.importances_std,
        ):
            importance_rows.append({
                "functional_group": fg,
                "model": model_name,
                "fold": fold,
                "process": "occurrence",
                "feature": feature,
                "importance_mean": imp_mean,
                "importance_sd": imp_sd,
            })

        # -------------------------------------------------------------
        # Positive abundance
        # -------------------------------------------------------------
        pos_train = y_train > 0
        pos_test = y_test > 0

        reg = clone(model_spec["regressor"])
        reg.fit(X_train[pos_train], y_train[pos_train])

        # Predict conditional abundance for all held-out rows so the
        # two components can be recombined as expected cover.
        mu_test_all = np.clip(reg.predict(X_test), 0.0, 1.0)
        oof_mu_positive[test_idx] = mu_test_all

        if pos_test.sum() >= 2:
            y_pos = y_test[pos_test]
            mu_pos = mu_test_all[pos_test]

            pos_r2 = r2_score(y_pos, mu_pos)
            pos_rmse = rmse(y_pos, mu_pos)
            pos_mae = mean_absolute_error(y_pos, mu_pos)

            pi_ab = permutation_importance(
                reg,
                X_test[pos_test],
                y_pos,
                scoring="r2",
                n_repeats=permutation_repeats,
                random_state=random_state + 100 + fold,
                n_jobs=-1,
            )

            for feature, imp_mean, imp_sd in zip(
                feature_cols,
                pi_ab.importances_mean,
                pi_ab.importances_std,
            ):
                importance_rows.append({
                    "functional_group": fg,
                    "model": model_name,
                    "fold": fold,
                    "process": "abundance",
                    "feature": feature,
                    "importance_mean": imp_mean,
                    "importance_sd": imp_sd,
                })
        else:
            pos_r2 = np.nan
            pos_rmse = np.nan
            pos_mae = np.nan

        expected_cover = p_test * mu_test_all

        fold_metrics.append({
            "functional_group": fg,
            "model": model_name,
            "fold": fold,
            "n_test": len(test_idx),
            "n_positive_test": int(pos_test.sum()),
            "occurrence_auc": auc,
            "occurrence_logloss": ll,
            "occurrence_brier": brier,
            "positive_r2": pos_r2,
            "positive_rmse": pos_rmse,
            "positive_mae": pos_mae,
            "hurdle_r2": r2_score(y_test, expected_cover),
            "hurdle_rmse": rmse(y_test, expected_cover),
        })

    expected_oof = oof_p_presence * oof_mu_positive
    pos_all = y > 0

    summary = {
        "functional_group": fg,
        "model": model_name,
        "n": len(work),
        "n_positive": int(pos_all.sum()),
        "zero_fraction": float((~pos_all).mean()),
        "oof_occurrence_auc": roc_auc_score(presence, oof_p_presence),
        "oof_occurrence_logloss": log_loss(
            presence, oof_p_presence, labels=[0, 1]
        ),
        "oof_occurrence_brier": brier_score_loss(
            presence, oof_p_presence
        ),
        "oof_positive_r2": r2_score(
            y[pos_all], oof_mu_positive[pos_all]
        ),
        "oof_positive_rmse": rmse(
            y[pos_all], oof_mu_positive[pos_all]
        ),
        "oof_positive_mae": mean_absolute_error(
            y[pos_all], oof_mu_positive[pos_all]
        ),
        "oof_hurdle_r2": r2_score(y, expected_oof),
        "oof_hurdle_rmse": rmse(y, expected_oof),
    }

    predictions = pd.DataFrame({
        "row_index": work.index,
        "cover": y,
        "presence": presence,
        "p_presence_oof": oof_p_presence,
        "mu_positive_oof": oof_mu_positive,
        "expected_cover_oof": expected_oof,
    })

    return (
        pd.DataFrame([summary]),
        pd.DataFrame(fold_metrics),
        pd.DataFrame(importance_rows),
        predictions,
    )

In [6]:
all_summary = []
all_fold_metrics = []
all_importance = []
all_predictions = []

for fg in FUNCTIONAL_GROUPS:
    for model_name, model_spec in MODELS.items():

        print(f"Running {model_name}: {fg}")

        summary_i, folds_i, importance_i, preds_i = fit_fg_cv(
            data=df,
            fg=fg,
            feature_cols=IA_COLS,
            model_name=model_name,
            model_spec=model_spec,
        )

        preds_i["functional_group"] = fg
        preds_i["model"] = model_name

        all_summary.append(summary_i)
        all_fold_metrics.append(folds_i)
        all_importance.append(importance_i)
        all_predictions.append(preds_i)

summary_results = pd.concat(all_summary, ignore_index=True)
fold_results = pd.concat(all_fold_metrics, ignore_index=True)
importance_results = pd.concat(all_importance, ignore_index=True)
prediction_results = pd.concat(all_predictions, ignore_index=True)

display(
    summary_results.sort_values(
        ["functional_group", "model"]
    )
)

Running RandomForest: BAREGROUND_FG
Running XGBoost: BAREGROUND_FG
Running RandomForest: BIOCRUST_FG
Running XGBoost: BIOCRUST_FG
Running RandomForest: EAG_FG
Running XGBoost: EAG_FG
Running RandomForest: EF_FG
Running XGBoost: EF_FG
Running RandomForest: LITTER_FG
Running XGBoost: LITTER_FG
Running RandomForest: PBG_FG
Running XGBoost: PBG_FG
Running RandomForest: SHRUB_total
Running XGBoost: SHRUB_total


,functional_group,model,n,n_positive,zero_fraction,oof_occurrence_auc,oof_occurrence_logloss,oof_occurrence_brier,oof_positive_r2,oof_positive_rmse,oof_positive_mae,oof_hurdle_r2,oof_hurdle_rmse
0,BAREGROUND_FG,RandomForest,489,481,0.016360,0.726091,0.145911,0.017848,0.585437,0.119112,0.089587,0.587590,0.119160
1,BAREGROUND_FG,XGBoost,489,481,0.016360,0.454002,0.083836,0.016102,0.624085,0.113424,0.085014,0.625118,0.113610
2,BIOCRUST_FG,RandomForest,489,384,0.214724,0.801091,0.424529,0.136079,0.265697,0.108362,0.083222,0.308725,0.102382
3,BIOCRUST_FG,XGBoost,489,384,0.214724,0.811682,0.453061,0.143091,0.251001,0.109441,0.080422,0.314467,0.101956
4,EAG_FG,RandomForest,489,346,0.292434,0.866365,0.414759,0.136287,0.517680,0.198059,0.161774,0.553158,0.193102
5,EAG_FG,XGBoost,489,346,0.292434,0.866284,0.441046,0.143332,0.525744,0.196397,0.152662,0.576443,0.188004
6,EF_FG,RandomForest,489,318,0.349693,0.738111,0.575641,0.195207,0.352131,0.139885,0.105269,0.286500,0.131888
7,EF_FG,XGBoost,489,318,0.349693,0.766560,0.570290,0.190942,0.359278,0.139111,0.100525,0.313647,0.129355
8,LITTER_FG,RandomForest,489,468,0.042945,0.748321,0.233752,0.043364,0.354829,0.083096,0.062586,0.371037,0.082526
9,LITTER_FG,XGBoost,489,468,0.042945,0.624440,0.191610,0.041575,0.362049,0.082629,0.062166,0.381481,0.081838


In [7]:
importance_summary = (
    importance_results
    .groupby(
        ["functional_group", "model", "process", "feature"],
        as_index=False
    )
    .agg(
        mean_importance=("importance_mean", "mean"),
        sd_importance=("importance_mean", "std"),
        median_importance=("importance_mean", "median"),
        positive_fold_fraction=(
            "importance_mean",
            lambda x: np.mean(np.asarray(x) > 0)
        ),
        n_folds=("fold", "nunique"),
    )
)

importance_summary["stability_score"] = (
    importance_summary["mean_importance"]
    * importance_summary["positive_fold_fraction"]
)

display(
    importance_summary
    .sort_values("stability_score", ascending=False)
    .head(50)
)

,functional_group,model,process,feature,mean_importance,sd_importance,median_importance,positive_fold_fraction,n_folds,stability_score
850,LITTER_FG,XGBoost,abundance,constant_BSI,0.270468,0.063836,0.300402,1.0,5,0.270468
299,BIOCRUST_FG,XGBoost,abundance,amp2_NDVI,0.184873,0.068522,0.171856,1.0,5,0.184873
998,PBG_FG,XGBoost,abundance,amp1_B8,0.160048,0.059242,0.138458,1.0,5,0.160048
131,BAREGROUND_FG,XGBoost,abundance,constant_NBR2,0.128107,0.023662,0.132081,1.0,5,0.128107
810,LITTER_FG,XGBoost,abundance,amp1_B11,0.110532,0.027157,0.098572,1.0,5,0.110532
497,EAG_FG,XGBoost,occurrence,amp1_B2,0.106309,0.027748,0.103282,1.0,5,0.106309
92,BAREGROUND_FG,XGBoost,abundance,amp1_B2,0.078710,0.014987,0.082705,1.0,5,0.078710
209,BIOCRUST_FG,RandomForest,abundance,amp2_NDVI,0.078032,0.020402,0.074646,1.0,5,0.078032
719,EF_FG,XGBoost,occurrence,constant_NDVI,0.062699,0.018871,0.066395,1.0,5,0.062699
452,EAG_FG,XGBoost,abundance,amp1_B2,0.062391,0.012147,0.060929,1.0,5,0.062391


## Correlated predictors

Do not interpret a single ranked feature as a unique causal effect.

Neighboring Sentinel-2 bands and derived indices are expected to be correlated. If two features contain substitutable information, a tree model may use one in one fold and the other in another fold.

Prioritize:

- importance stable across folds;
- agreement between Random Forest and XGBoost;
- recurrence across functional groups;
- recurrence within meaningful spectral families;
- grouped importance where strong correlation makes individual attribution unstable.

In [8]:
def feature_family(feature):
    f = feature.lower()

    if f.startswith(("constant_", "intercept_")):
        harmonic = "Intercept"
    elif f.startswith(("amp1_", "amplitude1_")):
        harmonic = "H1 amplitude"
    elif f.startswith(("amp2_", "amplitude2_")):
        harmonic = "H2 amplitude"
    else:
        harmonic = "Other"

    variable = feature.split("_", 1)[1] if "_" in feature else feature
    v = variable.upper()

    if v in {"B2", "B3", "B4"}:
        spectral = "Visible"
    elif v in {"B5", "B6", "B7"}:
        spectral = "Red edge"
    elif v in {"B8", "B8A"}:
        spectral = "NIR"
    elif v in {"B11", "B12"}:
        spectral = "SWIR"
    else:
        spectral = "Index"

    return harmonic, spectral


family_lookup = pd.DataFrame([
    {
        "feature": f,
        "harmonic_family": feature_family(f)[0],
        "spectral_family": feature_family(f)[1],
    }
    for f in IA_COLS
])

importance_with_family = importance_summary.merge(
    family_lookup,
    on="feature",
    how="left",
)

family_importance = (
    importance_with_family
    .groupby(
        [
            "functional_group",
            "model",
            "process",
            "harmonic_family",
            "spectral_family",
        ],
        as_index=False,
    )
    .agg(
        summed_mean_importance=("mean_importance", "sum"),
        mean_stability=("positive_fold_fraction", "mean"),
        n_features=("feature", "nunique"),
    )
    .sort_values("summed_mean_importance", ascending=False)
)

display(family_importance.head(50))

,functional_group,model,process,harmonic_family,spectral_family,summed_mean_importance,mean_stability,n_features
280,LITTER_FG,XGBoost,abundance,Intercept,Index,0.312283,0.720000,5
40,BAREGROUND_FG,XGBoost,abundance,Intercept,Index,0.216924,0.840000,5
95,BIOCRUST_FG,XGBoost,abundance,H2 amplitude,Index,0.176252,0.480000,5
273,LITTER_FG,XGBoost,abundance,H1 amplitude,SWIR,0.161895,1.000000,2
331,PBG_FG,XGBoost,abundance,H1 amplitude,NIR,0.152520,0.500000,2
10,BAREGROUND_FG,RandomForest,abundance,Intercept,Index,0.108142,1.000000,5
169,EAG_FG,XGBoost,occurrence,H1 amplitude,Visible,0.107531,0.666667,3
30,BAREGROUND_FG,XGBoost,abundance,H1 amplitude,Index,0.102141,0.880000,5
150,EAG_FG,XGBoost,abundance,H1 amplitude,Index,0.100478,0.880000,5
154,EAG_FG,XGBoost,abundance,H1 amplitude,Visible,0.090926,0.866667,3


## Important limitation of the family table

The family table above is only a **descriptive aggregation of individual permutation importances**.

The stronger follow-up is **true grouped permutation importance**: permute all members of a correlated information family together in held-out data and measure the performance loss.

Examples:

- all red-edge intercepts together;
- all H1 red-edge amplitudes together;
- all NIR amplitudes together;
- all index intercepts together.

That directly asks whether a correlated information family matters without forcing attribution to one member.

In [9]:
OUTPUT_DIR = DATA_CSV.parent / "RF_XGBoost_K2_IA_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_results.to_csv(
    OUTPUT_DIR / "model_summary.csv",
    index=False,
)

fold_results.to_csv(
    OUTPUT_DIR / "fold_metrics.csv",
    index=False,
)

importance_results.to_csv(
    OUTPUT_DIR / "fold_permutation_importance.csv",
    index=False,
)

importance_summary.to_csv(
    OUTPUT_DIR / "permutation_importance_summary.csv",
    index=False,
)

family_importance.to_csv(
    OUTPUT_DIR / "family_importance_summary.csv",
    index=False,
)

prediction_results.to_csv(
    OUTPUT_DIR / "oof_predictions.csv",
    index=False,
)

print(f"Results written to: {OUTPUT_DIR}")

Results written to: C:\NCA_DATA\Analysis\RF_XGBoost_K2_IA_results


## Recommended interpretation sequence

1. Compare Random Forest and XGBoost predictive performance for occurrence, positive abundance, and combined expected cover.
2. Inspect whether feature rankings are stable across folds.
3. Compare rankings between RF and XGBoost.
4. Look for recurring spectral **families**, not just single-band winners.
5. Add true grouped permutation importance for highly correlated families.
6. Use these results alongside PCA/ordination to define candidate clustering feature spaces.
7. Validate candidate spaces by ecological correspondence and cluster stability rather than feature importance alone.

In [10]:
# =============================================================================
# K2 IA SPECTRAL-FAMILY ABLATION
#
# 15 information families:
#   Intercept × Visible / Red edge / NIR / SWIR / Index
#   H1        × Visible / Red edge / NIR / SWIR / Index
#   H2        × Visible / Red edge / NIR / SWIR / Index
#
# For each family:
#   1. FAMILY ONLY:
#      How predictive is this information family by itself?
#
#   2. LEAVE FAMILY OUT:
#      How much performance is lost from the full IA model
#      when this family is removed?
#
# Models:
#   Random Forest
#   XGBoost
#
# Processes:
#   Occurrence
#   Positive abundance
#   Combined expected cover
# =============================================================================

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# DEFINE THE 15 SPECTRAL × HARMONIC FAMILIES
# -----------------------------------------------------------------------------

def get_variable(feature):
    """
    Strip harmonic prefix and return spectral variable name.
    """
    return feature.split("_", 1)[1]


def get_spectral_family(feature):
    """
    Assign a feature to one of five spectral information families.
    """

    variable = get_variable(feature).upper()

    if variable in {"B2", "B3", "B4"}:
        return "Visible"

    elif variable in {"B5", "B6", "B7"}:
        return "Red edge"

    elif variable in {"B8", "B8A"}:
        return "NIR"

    elif variable in {"B11", "B12"}:
        return "SWIR"

    else:
        return "Index"


def get_harmonic_family(feature):
    """
    Assign feature to intercept, H1 amplitude, or H2 amplitude.
    """

    f = feature.lower()

    if f.startswith(("constant_", "intercept_")):
        return "Intercept"

    elif f.startswith(("amp1_", "amplitude1_")):
        return "H1"

    elif f.startswith(("amp2_", "amplitude2_")):
        return "H2"

    else:
        raise ValueError(
            f"Unrecognized K2 IA feature: {feature}"
        )


family_membership = pd.DataFrame({
    "feature": IA_COLS
})

family_membership["harmonic_family"] = (
    family_membership["feature"]
    .map(get_harmonic_family)
)

family_membership["spectral_family"] = (
    family_membership["feature"]
    .map(get_spectral_family)
)

family_membership["family"] = (
    family_membership["harmonic_family"]
    + " × "
    + family_membership["spectral_family"]
)


FAMILY_COLS = {
    family: grp["feature"].tolist()
    for family, grp
    in family_membership.groupby(
        "family",
        sort=False
    )
}


print(
    f"{len(FAMILY_COLS)} spectral-information families\n"
)

for family, cols in FAMILY_COLS.items():

    print(
        f"{family:<24} "
        f"{len(cols):>2} predictors: "
        f"{', '.join(cols)}"
    )


# -----------------------------------------------------------------------------
# HELPER
#
# We reuse fit_fg_cv() from the existing notebook.
#
# permutation_repeats is set low here because feature permutation importance
# is NOT the objective of this experiment. We are comparing predictive
# performance after changing the entire feature set.
# -----------------------------------------------------------------------------

def run_feature_set(
    data,
    fg,
    model_name,
    model_spec,
    feature_cols,
    feature_set,
    family=None,
):

    summary_i, _, _, _ = fit_fg_cv(
        data=data,
        fg=fg,
        feature_cols=feature_cols,
        model_name=model_name,
        model_spec=model_spec,
        permutation_repeats=1,
    )

    summary_i["feature_set"] = feature_set
    summary_i["family"] = family
    summary_i["n_predictors"] = len(feature_cols)

    return summary_i


# -----------------------------------------------------------------------------
# RUN FULL BASELINE + FAMILY ONLY + LEAVE FAMILY OUT
# -----------------------------------------------------------------------------

family_ablation_results = []


for fg in FUNCTIONAL_GROUPS:

    print(
        "\n"
        + "=" * 80
    )

    print(
        fg
    )

    print(
        "=" * 80
    )


    for model_name, model_spec in MODELS.items():

        print(
            f"\n{model_name}"
        )


        # ---------------------------------------------------------------------
        # FULL K2 IA BASELINE
        # ---------------------------------------------------------------------

        print(
            "  Full K2 IA"
        )

        full_result = run_feature_set(
            data=df,
            fg=fg,
            model_name=model_name,
            model_spec=model_spec,
            feature_cols=IA_COLS,
            feature_set="Full IA",
            family="Full IA",
        )

        family_ablation_results.append(
            full_result
        )


        # ---------------------------------------------------------------------
        # EACH OF THE 15 FAMILIES
        # ---------------------------------------------------------------------

        for family, family_cols in FAMILY_COLS.items():

            # -------------------------------------------------------------
            # FAMILY ONLY
            # -------------------------------------------------------------

            print(
                f"  Family only:     {family}"
            )

            only_result = run_feature_set(
                data=df,
                fg=fg,
                model_name=model_name,
                model_spec=model_spec,
                feature_cols=family_cols,
                feature_set="Family only",
                family=family,
            )

            family_ablation_results.append(
                only_result
            )


            # -------------------------------------------------------------
            # LEAVE FAMILY OUT
            # -------------------------------------------------------------

            remaining_cols = [
                c for c in IA_COLS
                if c not in family_cols
            ]

            print(
                f"  Leave out:       {family}"
            )

            drop_result = run_feature_set(
                data=df,
                fg=fg,
                model_name=model_name,
                model_spec=model_spec,
                feature_cols=remaining_cols,
                feature_set="Leave family out",
                family=family,
            )

            family_ablation_results.append(
                drop_result
            )


family_ablation_results = pd.concat(
    family_ablation_results,
    ignore_index=True
)


display(
    family_ablation_results.head()
)

15 spectral-information families

Intercept × Visible       3 predictors: constant_B2, constant_B3, constant_B4
Intercept × Red edge      3 predictors: constant_B5, constant_B6, constant_B7
Intercept × NIR           2 predictors: constant_B8, constant_B8A
Intercept × SWIR          2 predictors: constant_B11, constant_B12
Intercept × Index         5 predictors: constant_NDVI, constant_NDMI, constant_NDRE, constant_BSI, constant_NBR2
H1 × Visible              3 predictors: amp1_B2, amp1_B3, amp1_B4
H1 × Red edge             3 predictors: amp1_B5, amp1_B6, amp1_B7
H1 × NIR                  2 predictors: amp1_B8, amp1_B8A
H1 × SWIR                 2 predictors: amp1_B11, amp1_B12
H1 × Index                5 predictors: amp1_NDVI, amp1_NDMI, amp1_NDRE, amp1_BSI, amp1_NBR2
H2 × Visible              3 predictors: amp2_B2, amp2_B3, amp2_B4
H2 × Red edge             3 predictors: amp2_B5, amp2_B6, amp2_B7
H2 × NIR                  2 predictors: amp2_B8, amp2_B8A
H2 × SWIR                 2 pred

,functional_group,model,n,n_positive,zero_fraction,oof_occurrence_auc,oof_occurrence_logloss,oof_occurrence_brier,oof_positive_r2,oof_positive_rmse,oof_positive_mae,oof_hurdle_r2,oof_hurdle_rmse,feature_set,family,n_predictors
0,BAREGROUND_FG,RandomForest,489,481,0.01636,0.726091,0.145911,0.017848,0.585437,0.119112,0.089587,0.587590,0.119160,Full IA,Full IA,45
1,BAREGROUND_FG,RandomForest,489,481,0.01636,0.428534,0.438455,0.024358,0.418262,0.141099,0.105964,0.410306,0.142489,Family only,Intercept × Visible,3
2,BAREGROUND_FG,RandomForest,489,481,0.01636,0.740125,0.143952,0.017577,0.569481,0.121382,0.092301,0.571958,0.121398,Leave family out,Intercept × Visible,42
3,BAREGROUND_FG,RandomForest,489,481,0.01636,0.284953,0.576885,0.025641,0.339966,0.150294,0.113221,0.324374,0.152518,Family only,Intercept × Red edge,3
4,BAREGROUND_FG,RandomForest,489,481,0.01636,0.738565,0.144131,0.017587,0.586511,0.118957,0.089428,0.588439,0.119038,Leave family out,Intercept × Red edge,42


In [12]:
# =============================================================================
# CALCULATE FAMILY SUFFICIENCY AND UNIQUE CONTRIBUTION
# =============================================================================


# -----------------------------------------------------------------------------
# Extract full-model baseline
# -----------------------------------------------------------------------------

full_baseline = (
    family_ablation_results[
        family_ablation_results["feature_set"] == "Full IA"
    ][
        [
            "functional_group",
            "model",
            "oof_occurrence_auc",
            "oof_positive_r2",
            "oof_hurdle_r2",
        ]
    ]
    .rename(
        columns={
            "oof_occurrence_auc":
                "full_occurrence_auc",

            "oof_positive_r2":
                "full_positive_r2",

            "oof_hurdle_r2":
                "full_hurdle_r2",
        }
    )
)


# -----------------------------------------------------------------------------
# FAMILY-ONLY PERFORMANCE
#
# Measures SUFFICIENCY:
#
# How much of the ecological signal can this family recover on its own?
# -----------------------------------------------------------------------------

family_only = (
    family_ablation_results[
        family_ablation_results["feature_set"] == "Family only"
    ]
    .merge(
        full_baseline,
        on=[
            "functional_group",
            "model",
        ],
        how="left",
    )
)


family_only["occurrence_fraction_of_full"] = (
    family_only["oof_occurrence_auc"] - 0.5
) / (
    family_only["full_occurrence_auc"] - 0.5
)


family_only["positive_r2_fraction_of_full"] = (
    family_only["oof_positive_r2"]
    /
    family_only["full_positive_r2"]
)


family_only["hurdle_r2_fraction_of_full"] = (
    family_only["oof_hurdle_r2"]
    /
    family_only["full_hurdle_r2"]
)


# -----------------------------------------------------------------------------
# LEAVE-FAMILY-OUT PERFORMANCE
#
# Measures UNIQUE / NON-SUBSTITUTABLE CONTRIBUTION:
#
# delta > 0 means performance becomes worse when the family is removed.
# -----------------------------------------------------------------------------

leave_out = (
    family_ablation_results[
        family_ablation_results["feature_set"]
        == "Leave family out"
    ]
    .merge(
        full_baseline,
        on=[
            "functional_group",
            "model",
        ],
        how="left",
    )
)


leave_out["delta_occurrence_auc"] = (
    leave_out["full_occurrence_auc"]
    -
    leave_out["oof_occurrence_auc"]
)


leave_out["delta_positive_r2"] = (
    leave_out["full_positive_r2"]
    -
    leave_out["oof_positive_r2"]
)


leave_out["delta_hurdle_r2"] = (
    leave_out["full_hurdle_r2"]
    -
    leave_out["oof_hurdle_r2"]
)


# -----------------------------------------------------------------------------
# COMBINED FAMILY COMPARISON TABLE
# -----------------------------------------------------------------------------

family_comparison = (
    family_only[
        [
            "functional_group",
            "model",
            "family",
            "n_predictors",

            "oof_occurrence_auc",
            "occurrence_fraction_of_full",

            "oof_positive_r2",
            "positive_r2_fraction_of_full",

            "oof_hurdle_r2",
            "hurdle_r2_fraction_of_full",
        ]
    ]
    .rename(
        columns={
            "oof_occurrence_auc":
                "family_only_occurrence_auc",

            "oof_positive_r2":
                "family_only_positive_r2",

            "oof_hurdle_r2":
                "family_only_hurdle_r2",
        }
    )
    .merge(
        leave_out[
            [
                "functional_group",
                "model",
                "family",

                "delta_occurrence_auc",
                "delta_positive_r2",
                "delta_hurdle_r2",
            ]
        ],
        on=[
            "functional_group",
            "model",
            "family",
        ],
        how="left",
    )
)

family_comparison_sorted = (
    family_comparison
    .sort_values(
        [
            "functional_group",
            "model",
            "delta_hurdle_r2",
        ],
        ascending=[
            True,
            True,
            False,
        ]
    )
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(family_comparison_sorted)

family_comparison_sorted.to_csv(
    OUTPUT_DIR / "spectral_family_comparison_full.csv",
    index=False,
)

,functional_group,model,family,n_predictors,family_only_occurrence_auc,occurrence_fraction_of_full,family_only_positive_r2,positive_r2_fraction_of_full,family_only_hurdle_r2,hurdle_r2_fraction_of_full,delta_occurrence_auc,delta_positive_r2,delta_hurdle_r2
0,BAREGROUND_FG,RandomForest,Intercept × Index,5,0.741944,1.070115,0.508762,0.869030,0.512716,0.872575,-0.025598,0.028361,0.029466
1,BAREGROUND_FG,RandomForest,Intercept × Visible,3,0.428534,-0.316092,0.418262,0.714445,0.410306,0.698285,-0.014033,0.015955,0.015632
2,BAREGROUND_FG,RandomForest,H1 × Visible,3,0.554574,0.241379,0.303086,0.517708,0.306361,0.521385,-0.008706,0.013310,0.012383
3,BAREGROUND_FG,RandomForest,H1 × Index,5,0.419179,-0.357471,0.283600,0.484425,0.287952,0.490056,-0.026377,0.007581,0.007658
4,BAREGROUND_FG,RandomForest,H2 × Visible,3,0.478170,-0.096552,-0.018899,-0.032283,-0.037858,-0.064430,-0.023649,0.003907,0.004209
5,BAREGROUND_FG,RandomForest,H1 × SWIR,2,0.206861,-1.296552,0.015388,0.026284,0.018686,0.031801,-0.004548,0.003025,0.003762
6,BAREGROUND_FG,RandomForest,H2 × Index,5,0.717516,0.962069,0.038133,0.065136,0.056380,0.095951,0.040021,0.001322,0.001582
7,BAREGROUND_FG,RandomForest,Intercept × NIR,2,0.459719,-0.178161,0.180609,0.308502,0.173822,0.295822,-0.002469,0.000923,0.000966
8,BAREGROUND_FG,RandomForest,H2 × NIR,2,0.464527,-0.156897,-0.124774,-0.213130,-0.092742,-0.157834,-0.016502,0.000290,0.000842
9,BAREGROUND_FG,RandomForest,Intercept × SWIR,2,0.745712,1.086782,0.371455,0.634492,0.375221,0.638576,0.014293,0.000591,0.000636
